## ⚙️ Setup Instructions

Before running this notebook:

### 1. Install dependencies (if not already installed):
```bash
pip install pyyaml catboost scikit-learn shap optuna tqdm
```

### 2. Place input data files:
- `data/TOPMed_stopgain.csv` — main variant CSV
- `data/annotations/codon_optimality.tsv`
- `data/annotations/readthrough.csv`
- `data/annotations/ejc_occupancy.tsv`
- `data/annotations/ptc_aug.tsv`
- `data/annotations/conservation_medians.csv`

See `data/README.md` for instructions on obtaining these files.

### 3. Ensure directory structure:
```
repo_root/
├── config/
│   └── config.yaml
├── data/
│   ├── TOPMed_stopgain.csv
│   └── annotations/
├── notebooks/
│   └── 01_data_loading_and_merging.ipynb    ← You are here
└── results/                                 ← created automatically
```

### 4. Run cells in order!

---

## 📁 Output

This notebook produces `data/TOPMed_merged.csv` as input for Notebook 02.

---

## 🚨 Troubleshooting

**Error: "FileNotFoundError: config.yaml"**
- Check that `config/config.yaml` exists one level above `notebooks/`

**Error: "FileNotFoundError: [data file]"**
- Check that your files are named to match the paths in `config/config.yaml`
- See `data/README.md` for sourcing instructions

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import yaml

## Configuration

In [ ]:
# Load configuration
config_path = Path("../config/config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

# Anchor all relative paths to the repo root
BASE_DIR = config_path.resolve().parent.parent

# Input paths from config
PATH_ENHANCED     = BASE_DIR / config['data']['enhanced_features']
PATH_CODON_OPT    = BASE_DIR / config['data']['codon_optimality']
PATH_READTHROUGH  = BASE_DIR / config['data']['readthrough']
PATH_EJC          = BASE_DIR / config['data']['ejc']
PATH_PTCAUG       = BASE_DIR / config['data']['ptc_aug']
PATH_CONSERVATION = BASE_DIR / config['data']['conservation']

# Output path from config
PATH_OUTPUT = BASE_DIR / config['data']['merged']
PATH_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

print("✓ Configuration loaded from:", config_path.resolve())
print(f"\nInput files:")
print(f"  Enhanced:      {PATH_ENHANCED}")
print(f"  Codon opt:     {PATH_CODON_OPT}")
print(f"  Readthrough:   {PATH_READTHROUGH}")
print(f"  EJC:           {PATH_EJC}")
print(f"  PTC AUG:       {PATH_PTCAUG}")
print(f"  Conservation:  {PATH_CONSERVATION}")
print(f"\nOutput:")
print(f"  {PATH_OUTPUT}")

## 1. Load All Datasets

In [ ]:
print("="*80)
print("LOADING DATASETS")
print("="*80)

# Load enhanced features (main variant CSV)
df_enhanced = pd.read_csv(PATH_ENHANCED)
print(f"✓ Enhanced features: {df_enhanced.shape}")

# Load codon optimality data
df_codon = pd.read_csv(PATH_CODON_OPT, sep='\t')
print(f"✓ Codon optimality: {df_codon.shape}")

# Load readthrough data
df_readthrough = pd.read_csv(PATH_READTHROUGH)
print(f"✓ Readthrough: {df_readthrough.shape}")

# Load EJC occupancy data
df_ejc = pd.read_csv(PATH_EJC, sep='\t')
print(f"✓ EJC occupancy: {df_ejc.shape}")

# Load PTC AUG analysis
df_ptcaug = pd.read_csv(PATH_PTCAUG, sep='\t')
print(f"✓ PTC AUG: {df_ptcaug.shape}")

# Load conservation scores
df_conservation = pd.read_csv(PATH_CONSERVATION)
print(f"✓ Conservation: {df_conservation.shape}")

## 2. Remove Indels and Standardize Keys

In [ ]:
print("\n" + "="*80)
print("IDENTIFYING INDEL KEYS FROM READTHROUGH DATA")
print("="*80)

indel_keys = set()

if 'V7' in df_readthrough.columns and 'V8' in df_readthrough.columns:
    print(f"\nScanning readthrough data for indels...")
    
    # Identify rows with indels
    indel_mask = ((df_readthrough['V7'].astype(str).str.contains('-', na=False)) | 
                  (df_readthrough['V8'].astype(str).str.contains('-', na=False)))
    
    indels_v7 = df_readthrough['V7'].astype(str).str.contains('-', na=False).sum()
    indels_v8 = df_readthrough['V8'].astype(str).str.contains('-', na=False).sum()
    total_indels = indel_mask.sum()
    
    print(f"  Indels detected:")
    print(f"    V7 column: {indels_v7}")
    print(f"    V8 column: {indels_v8}")
    print(f"    Total rows with indels: {total_indels}")
    
    # Extract variantIDs of indel variants
    indel_keys = set(df_readthrough.loc[indel_mask, 'variantID'].values)
    
    print(f"\n✓ Identified {len(indel_keys)} unique indel variantIDs")
    print(f"  Sample indel keys: {list(indel_keys)[:3]}")
else:
    print("  ⚠️  V7/V8 columns not found - cannot identify indels")

print("\n" + "="*80)
print("FILTERING INDELS FROM ALL DATASETS")
print("="*80)

if len(indel_keys) > 0:
    # Filter enhanced dataset
    before = len(df_enhanced)
    df_enhanced = df_enhanced[~df_enhanced['variantID'].isin(indel_keys)]
    removed = before - len(df_enhanced)
    print(f"\nEnhanced dataset:")
    print(f"  Before: {before} variants")
    print(f"  After: {len(df_enhanced)} variants")
    print(f"  Removed: {removed} indels")
    
    # Filter codon optimality dataset based on PTC_ID format
    if 'PTC_ID' in df_codon.columns:
        before = len(df_codon)
        
        # Parse PTC_ID to check ref and alt allele lengths
        # Format: chr_pos_ref_alt
        def is_snv(ptc_id):
            parts = ptc_id.split('_')
            if len(parts) == 4:
                ref = parts[2]
                alt = parts[3]
                # SNV = both ref and alt are single nucleotides
                return len(ref) == 1 and len(alt) == 1
            return False
        
        snv_mask = df_codon['PTC_ID'].apply(is_snv)
        indels_in_codon = (~snv_mask).sum()
        
        print(f"\nCodon optimality - filtering by allele length:")
        print(f"  Total variants: {before}")
        print(f"  SNVs (len=1 for both ref/alt): {snv_mask.sum()}")
        print(f"  Indels (len>1 for ref or alt): {indels_in_codon}")
        
        if indels_in_codon > 0:
            print(f"  Sample indel PTC_IDs: {df_codon.loc[~snv_mask, 'PTC_ID'].head(3).tolist()}")
        
        df_codon = df_codon[snv_mask]
        removed = before - len(df_codon)
        
        print(f"  After filtering: {len(df_codon)} variants")
        print(f"  Removed: {removed} indels")
        
        # Create variantID column from PTC_ID (already in chr_pos_ref_alt format)
        df_codon['variantID'] = df_codon['PTC_ID']
        print(f"  ✓ variantID set from PTC_ID for {len(df_codon)} SNVs")
        print(f"  Sample variantIDs: {df_codon['variantID'].head(3).tolist()}")
    
    # Filter readthrough dataset
    before = len(df_readthrough)
    df_readthrough = df_readthrough[~df_readthrough['variantID'].isin(indel_keys)]
    removed = before - len(df_readthrough)
    print(f"\nReadthrough dataset:")
    print(f"  Before: {before} variants")
    print(f"  After: {len(df_readthrough)} variants")
    print(f"  Removed: {removed} indels")

    # Filter EJC dataset
    before = len(df_ejc)
    df_ejc = df_ejc.rename(columns={'PTC_ID': 'variantID'})
    df_ejc = df_ejc[~df_ejc['variantID'].isin(indel_keys)]
    removed = before - len(df_ejc)
    print(f"\nEJC dataset:")
    print(f"  Before: {before} variants")
    print(f"  After: {len(df_ejc)} variants")
    print(f"  Removed: {removed} indels")

    # Filter PTC AUG dataset
    before = len(df_ptcaug)
    df_ptcaug = df_ptcaug.rename(columns={'variant_id': 'variantID'})
    df_ptcaug = df_ptcaug[~df_ptcaug['variantID'].isin(indel_keys)]
    removed = before - len(df_ptcaug)
    print(f"\nPTC AUG dataset:")
    print(f"  Before: {before} variants")
    print(f"  After: {len(df_ptcaug)} variants")
    print(f"  Removed: {removed} indels")
    
    print(f"\n✓ Successfully filtered indel variants from all datasets")
else:
    print("\n  No indels to filter")
    # Still need to standardize keys even if no indels
    df_codon   = df_codon.rename(columns={'PTC_ID': 'variantID'})
    df_ejc     = df_ejc.rename(columns={'PTC_ID': 'variantID'})
    df_ptcaug  = df_ptcaug.rename(columns={'variant_id': 'variantID'})

## 3. Remove Old Features to be Replaced

In [ ]:
print("\n" + "="*80)
print("REMOVING OLD FEATURES")
print("="*80)

# Drop AverageCodonRNAUsage if present (replaced by codon optimality)
if 'AverageCodonRNAUsage' in df_enhanced.columns:
    print(f"\n✓ Dropping AverageCodonRNAUsage (replaced by codon optimality)")
    df_enhanced = df_enhanced.drop(columns=['AverageCodonRNAUsage'])
else:
    print(f"\n⚠️  AverageCodonRNAUsage not found (already dropped or not present)")

# Drop median_half_life if present (half-life now included in main CSV as half_life_PC1)
if 'median_half_life' in df_enhanced.columns:
    print(f"✓ Dropping median_half_life (replaced by half_life_PC1 in main CSV)")
    df_enhanced = df_enhanced.drop(columns=['median_half_life'])
else:
    print(f"⚠️  median_half_life not found (already dropped or not present)")

print(f"\nEnhanced dataset after drops: {df_enhanced.shape}")

## 4. Merge Codon Optimality Features

In [ ]:
print("\n" + "="*80)
print("MERGING CODON OPTIMALITY FEATURES")
print("="*80)

# Select columns to merge
codon_cols = ['variantID', 'CodonOptimalityFraction_CDS', 'CodonOptimalityFraction_PTCpm100nt']
print(f"\nMerging columns: {codon_cols}")

# Check which variants will match
common_keys = set(df_enhanced['variantID']) & set(df_codon['variantID'])
print(f"\nKey overlap: {len(common_keys)}/{len(df_enhanced)} variants")

# Merge
df_result = df_enhanced.merge(
    df_codon[codon_cols],
    on='variantID',
    how='left'
)

print(f"\n✓ Merged codon optimality features")
print(f"  Shape after merge: {df_result.shape}")

# Check merge results
for col in ['CodonOptimalityFraction_CDS', 'CodonOptimalityFraction_PTCpm100nt']:
    matched = df_result[col].notna().sum()
    pct = matched/len(df_result)*100
    print(f"  {col}: {matched}/{len(df_result)} ({pct:.1f}%)")
    if matched > 0:
        print(f"    Mean: {df_result[col].mean():.4f}")
        print(f"    Range: {df_result[col].min():.4f} - {df_result[col].max():.4f}")

## 5. Merge Readthrough Features

In [ ]:
print("\n" + "="*80)
print("MERGING READTHROUGH FEATURES")
print("="*80)

# Only pull new columns not already in main CSV
main_cols = set(df_enhanced.columns)
rt_new_cols = ['variantID'] + [c for c in df_readthrough.columns
               if c not in main_cols and c != 'variantID']

readthrough_feature_cols = [c for c in rt_new_cols if 'readthrough' in c.lower()]
print(f"\nNew readthrough feature columns: {readthrough_feature_cols}")
print(f"Total new columns to merge: {len(rt_new_cols) - 1}")

# Merge readthrough features
cols_to_merge = ['variantID', 'readthrough_score_hek293t', 'readthrough_category_hek293t']
cols_to_merge = [c for c in cols_to_merge if c in df_readthrough.columns]

print(f"Merging columns: {cols_to_merge}")

df_result = df_result.merge(
    df_readthrough[cols_to_merge],
    on='variantID',
    how='left'
)

print(f"\n✓ Merged readthrough features")
print(f"  Shape after merge: {df_result.shape}")

# Check merge results
for col in cols_to_merge:
    if col != 'variantID':
        matched = df_result[col].notna().sum()
        pct = matched/len(df_result)*100
        print(f"  {col}: {matched}/{len(df_result)} ({pct:.1f}%)")
        
        if 'category' in col:
            print(f"    Distribution:")
            print(df_result[col].value_counts().to_string(header=False))
        elif matched > 0:
            print(f"    Mean: {df_result[col].mean():.2f}")
            print(f"    Range: {df_result[col].min():.2f} - {df_result[col].max():.2f}")

## 6. Merge EJC Occupancy Features

In [ ]:
print("\n" + "="*80)
print("MERGING EJC OCCUPANCY FEATURES")
print("="*80)

# Only pull new columns not already in main CSV
ejc_new_cols = ['variantID'] + [c for c in df_ejc.columns
                if c not in main_cols and c != 'variantID']

print(f"\nEJC columns to merge: {[c for c in ejc_new_cols if c != 'variantID']}")

# Check overlap
common_keys = set(df_result['variantID']) & set(df_ejc['variantID'])
print(f"Key overlap: {len(common_keys)}/{len(df_result)} variants")

df_result = df_result.merge(
    df_ejc[ejc_new_cols],
    on='variantID',
    how='left'
)

print(f"\n✓ Merged EJC occupancy features")
print(f"  Shape after merge: {df_result.shape}")

for col in ['ejc_count_in_window', 'has_ejc_overlap']:
    if col in df_result.columns:
        matched = df_result[col].notna().sum()
        pct = matched/len(df_result)*100
        print(f"  {col}: {matched}/{len(df_result)} ({pct:.1f}%)")

## 7. AUG Feature Engineering

In [ ]:
print("\n" + "="*80)
print("STEP: Creating Enhanced AUG Features")
print("="*80)

# Analyze missingness in AUG data
aug_distance_na = df_ptcaug['nearest_inframe_aug_distance_nt'].isna().sum()
aug_kozak_na = df_ptcaug['nearest_inframe_kozak_strength'].isna().sum()
print(f"\nIn AUG dataframe:")
print(f"  Distance NA: {aug_distance_na}/{len(df_ptcaug)} ({aug_distance_na/len(df_ptcaug)*100:.1f}%)")
print(f"  Kozak NA:    {aug_kozak_na}/{len(df_ptcaug)} ({aug_kozak_na/len(df_ptcaug)*100:.1f}%)")
print(f"  → These are variants with NO in-frame AUG (biologically meaningful!)")

# 1. Numeric distance (keep as-is, NA = no AUG)
df_ptcaug['aug_distance_nt'] = df_ptcaug['nearest_inframe_aug_distance_nt']

# 2. Distance categories (explicit 'no_inframe_AUG')
def categorize_distance(dist):
    if pd.isna(dist):
        return 'no_inframe_AUG'
    elif dist < 50:
        return 'very_close'
    elif dist < 100:
        return 'close'
    elif dist < 200:
        return 'moderate'
    elif dist < 500:
        return 'far'
    else:
        return 'very_far'

df_ptcaug['aug_distance_category'] = df_ptcaug['aug_distance_nt'].apply(categorize_distance)
print(f"\nDistance categories created:")
print(df_ptcaug['aug_distance_category'].value_counts().sort_index())

# 3. Kozak strength — downstream/reinitiation AUG context (explicit 'no_inframe_AUG')
df_ptcaug['kozak_strength'] = df_ptcaug['nearest_inframe_kozak_strength'].fillna('no_inframe_AUG').astype(str)
print(f"\nKozak strength categories:")
print(df_ptcaug['kozak_strength'].value_counts().sort_index())

# 4. Frame AUGs (explicit False for NA)
df_ptcaug['has_plus1_aug'] = df_ptcaug['has_plus1_frame_aug'].fillna(False).astype(str)
df_ptcaug['has_plus2_aug'] = df_ptcaug['has_plus2_frame_aug'].fillna(False).astype(str)

# 5. Combined AUG frame status
def get_aug_frame_status(row):
    plus1 = str(row['has_plus1_aug']) == 'True'
    plus2 = str(row['has_plus2_aug']) == 'True'
    
    if plus1 and plus2:
        return 'both_frames'
    elif plus1:
        return 'plus1_only'
    elif plus2:
        return 'plus2_only'
    else:
        return 'no_frame_AUG'

df_ptcaug['aug_frame_status'] = df_ptcaug.apply(get_aug_frame_status, axis=1)
print(f"\nAUG frame status:")
print(df_ptcaug['aug_frame_status'].value_counts().sort_index())

## 8. Merge PTC AUG Features

In [ ]:
print("\n" + "="*80)
print("MERGING PTC AUG FEATURES")
print("="*80)

# Select engineered features + useful raw features to keep
aug_keep = [
    'variantID',
    # Engineered features (v3.2 compatible)
    'aug_distance_category',
    'aug_distance_nt',
    'kozak_strength',
    'has_plus1_aug',
    'has_plus2_aug',
    'aug_frame_status',
]
# Only keep columns that actually exist
aug_keep = [c for c in aug_keep if c in df_ptcaug.columns]

print(f"\nAUG columns to merge: {[c for c in aug_keep if c != 'variantID']}")

# Check overlap
common_keys = set(df_result['variantID']) & set(df_ptcaug['variantID'])
print(f"Key overlap: {len(common_keys)}/{len(df_result)} variants")

df_result = df_result.merge(
    df_ptcaug[aug_keep],
    on='variantID',
    how='left'
)

print(f"\n✓ Merged PTC AUG features")
print(f"  Shape after merge: {df_result.shape}")

for col in ['aug_distance_category', 'kozak_strength', 'aug_frame_status']:
    if col in df_result.columns:
        matched = df_result[col].notna().sum()
        print(f"  {col}: {matched}/{len(df_result)} non-null")

## 9. Merge Conservation Score Features

In [ ]:
print("\n" + "="*80)
print("MERGING CONSERVATION SCORE FEATURES")
print("="*80)

# Conservation CSV is a passthrough — only pull the new phastcons/phylop columns
cons_new_cols = ['variantID'] + [c for c in df_conservation.columns
                 if c.startswith('phastcons_') or c.startswith('phylop_')]

print(f"\nConservation columns to merge: {len(cons_new_cols) - 1}")
print(f"  PhastCons: {[c for c in cons_new_cols if c.startswith('phastcons_')]}")
print(f"  PhyloP:    {[c for c in cons_new_cols if c.startswith('phylop_')]}")

# Check overlap
common_keys = set(df_result['variantID']) & set(df_conservation['variantID'])
print(f"\nKey overlap: {len(common_keys)}/{len(df_result)} variants")

df_result = df_result.merge(
    df_conservation[cons_new_cols],
    on='variantID',
    how='left'
)

print(f"\n✓ Merged conservation features")
print(f"  Shape after merge: {df_result.shape}")

# Check a sample conservation column
sample_col = 'phastcons_ptc_100bp_median'
if sample_col in df_result.columns:
    matched = df_result[sample_col].notna().sum()
    print(f"  {sample_col}: {matched}/{len(df_result)} non-null ({matched/len(df_result)*100:.1f}%)")

## 10. Final Verification

In [ ]:
print("\n" + "="*80)
print("FINAL VERIFICATION")
print("="*80)

print(f"\nFinal dataset shape: {df_result.shape}")
print(f"  Original enhanced: {df_enhanced.shape[1]} columns")
print(f"  Final: {df_result.shape[1]} columns")
print(f"  Net change: {df_result.shape[1] - df_enhanced.shape[1]:+d} columns")

# Verify new features are present
print("\n✓ New features added:")
new_features_expected = [
    'CodonOptimalityFraction_CDS',
    'CodonOptimalityFraction_PTCpm100nt',
    'readthrough_score_hek293t',
    'readthrough_category_hek293t',
    'ejc_count_in_window',
    'has_ejc_overlap',
    'aug_distance_category',
    'kozak_strength',
    'aug_frame_status',
    'has_plus1_aug',
    'has_plus2_aug',
    'phastcons_ptc_100bp_median',
    'phylop_ptc_100bp_median',
    'half_life_PC1'
]
for col in new_features_expected:
    if col in df_result.columns:
        non_null = df_result[col].notna().sum()
        print(f"  ✔ {col}: {non_null}/{len(df_result)} non-null")
    else:
        print(f"  ✘ {col}: MISSING")

# Check for any NaN issues
print(f"\nMissing data summary:")
missing_summary = df_result.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
if len(missing_summary) > 0:
    print(f"  Columns with missing data: {len(missing_summary)}")
    print(f"  Top 10 columns with most missing:")
    for col, count in missing_summary.head(10).items():
        pct = count/len(df_result)*100
        print(f"    {col}: {count} ({pct:.1f}%)")
else:
    print("  No missing data!")

## 11. Save Merged Dataset

In [ ]:
print("\n" + "="*80)
print("SAVING MERGED DATASET")
print("="*80)

df_result.to_csv(PATH_OUTPUT, index=False)
print(f"\n✓ Saved: {PATH_OUTPUT}")
print(f"  Rows: {len(df_result)}")
print(f"  Columns: {df_result.shape[1]}")

print("\n" + "="*80)
print("COMPLETE! 🎉")
print("="*80)
print("\nReady for feature cleaning and selection (next notebook)")